# GS/BIOT 5050 Project Log Group 1: Clinical Trial Terminations (2015-2024)
AbdulRafeh, Muhammad Fawad Alam, Avneet, Fatiah
**Project Overview:** This notebook documents the complete, reproducible data pipeline for analyzing industry-sponsored Phase II and Phase III clinical trial terminations. This project replicates and extends the methodology of Bowling et al. (2025), *Analysis of phase II and phase III clinical trial terminations from 2013 to 2023* (Nature Reviews Drug Discovery).

## Instructor Run Notes
1. Open this notebook and select a Python kernel.
2. Click **Run All**.
3. The notebook will attempt to auto-fetch raw data from ClinicalTrials.gov, clean/classify it, and generate all figures.

**One caveat (important):** If your kernel has no `pip` access or package installation is blocked by policy, auto-install may fail. In that case, ensure `pandas`, `numpy`, and `matplotlib` are already installed, then click **Run All** again.

## Section 0: Setup and Dependencies
The following libraries are required to run this pipeline:
* `urllib`, `json`, `csv`, `ssl`, `time`, `re` (Python Standard Library)
* `pandas`, `numpy` (Data manipulation)
* `matplotlib` (Visualization)

In [ ]:
import os
import sys
import subprocess
import importlib
import warnings

# Auto-install only if packages are missing so "Run All" is one-click friendly.
def ensure_packages(packages):
    missing = []
    for p in packages:
        try:
            importlib.import_module(p)
        except ImportError:
            missing.append(p)

    if missing:
        print(f"Installing missing packages: {', '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

ensure_packages(["pandas", "numpy", "matplotlib"])

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import urllib.request
import urllib.parse
import json
import csv
import time
import ssl
import re

warnings.filterwarnings('ignore')
print(f"Python executable: {sys.executable}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")

## Section 1: Automated Data Acquisition
**Objective:** Fetch all relevant terminated trials directly from the ClinicalTrials.gov API v2.

**Search Strategy:**
The exact Expert Search query used is:
`AREA[OverallStatus]TERMINATED AND AREA[Phase](PHASE2 OR PHASE3) AND AREA[LeadSponsorClass]INDUSTRY AND AREA[PrimaryCompletionDate]RANGE[2015-01-01,2025-01-01]`

*Methodological Note on Date Field:* `PrimaryCompletionDate` was selected over `StudyFirstSubmitDate` because it represents the actual chronological point when data collection for the primary outcome measure ceased, which most accurately reflects the true termination timeline.

*Methodological Note on API Handling:* The modernized API v2 requires the `query.term` parameter for Expert Search strings. A global SSL bypass is included in this script to ensure the automated pull does not fail on macOS environments with strict certificate verification.

In [ ]:
# Step 1: Fetch raw data with one-click behavior for "Run All".
# Default behavior:
# - Tries to fetch fresh data from API
# - Falls back to local CSV if API is unavailable

import os

RAW_CSV_PATH = "raw_trials_with_reason.csv"
AUTO_FETCH_FROM_API = True
ALLOW_FALLBACK_TO_LOCAL = True

def fetch_clinical_trials_data(output_csv=RAW_CSV_PATH):
    # Bypass Mac SSL verification
    try:
        _create_unverified_https_context = ssl._create_unverified_context
    except AttributeError:
        pass
    else:
        ssl._create_default_https_context = _create_unverified_https_context

    print("Starting automated data acquisition from ClinicalTrials.gov API v2...")
    query = "AREA[OverallStatus]TERMINATED AND AREA[Phase](PHASE2 OR PHASE3) AND AREA[LeadSponsorClass]INDUSTRY AND AREA[PrimaryCompletionDate]RANGE[2015-01-01,2025-01-01]"
    encoded_query = urllib.parse.quote(query)
    base_url = f"https://clinicaltrials.gov/api/v2/studies?query.term={encoded_query}&pageSize=1000"

    all_trials = []
    next_page_token = None

    while True:
        url = base_url if not next_page_token else base_url + f"&pageToken={next_page_token}"
        req = urllib.request.Request(url, headers={"User-Agent": "Python/urllib"})

        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode("utf-8"))

        studies = data.get("studies", [])
        all_trials.extend(studies)

        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break
        time.sleep(0.5)

    if len(all_trials) == 0:
        raise RuntimeError("API returned zero studies; raw CSV not written.")

    print(f"Success! Fetched {len(all_trials)} trials.")

    headers = [
        "nct_id", "study_title", "study_status", "phase", "sponsor", "funder_type",
        "conditions", "interventions", "enrollment", "start_date", "primary_completion_date",
        "completion_date", "first_posted", "why_stopped", "study_type", "study_design"
    ]

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        csv.field_size_limit(10_000_000)
        writer = csv.writer(f)
        writer.writerow(headers)

        for study in all_trials:
            protocol = study.get("protocolSection", {})
            id_mod = protocol.get("identificationModule", {})
            status_mod = protocol.get("statusModule", {})
            design_mod = protocol.get("designModule", {})
            sponsor_mod = protocol.get("sponsorCollaboratorsModule", {})

            nct_id = id_mod.get("nctId", "")
            title = id_mod.get("briefTitle", "")
            status = status_mod.get("overallStatus", "")
            phases = design_mod.get("phases", [])
            phase_str = "|".join(phases) if phases else ""
            sponsor = sponsor_mod.get("leadSponsor", {}).get("name", "")
            funder = sponsor_mod.get("leadSponsor", {}).get("class", "")
            enrollment = design_mod.get("enrollmentInfo", {}).get("count", "")
            prim_comp_date = status_mod.get("primaryCompletionDateStruct", {}).get("date", "")
            why_stopped = status_mod.get("whyStopped", "")

            writer.writerow([
                nct_id, title, status, phase_str, sponsor, funder, "", "",
                enrollment, "", prim_comp_date, "", "", why_stopped, "", ""
            ])

    return output_csv

# One-click data preparation logic
if AUTO_FETCH_FROM_API:
    try:
        fetch_clinical_trials_data(RAW_CSV_PATH)
    except Exception as e:
        print(f"API fetch failed: {e}")
        if ALLOW_FALLBACK_TO_LOCAL and os.path.exists(RAW_CSV_PATH):
            print(f"Falling back to existing local file: {RAW_CSV_PATH}")
        else:
            raise RuntimeError(
                "Could not fetch data from API and no local raw CSV is available. "
                "Place raw_trials_with_reason.csv next to this notebook or check network/API access."
            )

# Load the raw data for downstream steps
raw_df = pd.read_csv(RAW_CSV_PATH)
print(f"Raw data shape: {raw_df.shape}")

## Section 2: Data Tidying and Classification
**Objective:** Clean the raw data, extract the completion year, and classify the free-text `why_stopped` reasons into the 6 categories defined by Bowling et al. (2025).

**Classification Hierarchy (First-match wins):**
1. **Safety:** adverse, toxicity, harm, death, SAE.
2. **Efficacy:** efficacy, lack of benefit, futility, endpoint not met.
3. **Enrolment:** enrollment, recruit, screen failure.
4. **Strategy/Business:** business, commercial, merger, funding withdrawn, clinical hold.
5. **Operational:** logistic, protocol issue, covid, site closure.
6. **Unknown:** Blank fields or no keyword match.

In [ ]:
def clean_and_classify_data(df):
    # Standardize missing values
    df = df.replace(['', 'nan', 'None', 'N/A', 'n/a', 'NaN'], np.nan)
    
    # Extract 4-digit year
    def extract_year(date_str):
        if pd.isna(date_str): return np.nan
        match = re.search(r'\b(201[5-9]|202[0-5])\b', str(date_str))
        return int(match.group(1)) if match else np.nan
    df['year'] = df['primary_completion_date'].apply(extract_year)
    
    # Keyword Classifier
    def classify(reason):
        if pd.isna(reason): return 'Unknown'
        r = str(reason).lower()
        if any(k in r for k in ['safety', 'adverse', 'toxicity', 'harm', 'sae', 'death', 'mortality', 'tolerability']): return 'Safety'
        if any(k in r for k in ['efficacy', 'effectiveness', 'lack of effect', 'lack of benefit', 'futility', 'endpoint']): return 'Efficacy'
        if any(k in r for k in ['enrollment', 'enrolment', 'recruit', 'accru', 'screen failure', 'eligib']): return 'Enrolment'
        if any(k in r for k in ['business', 'strateg', 'commercial', 'portfolio', 'sponsor decision', 'merger', 'funding withdrawn']): return 'Strategy/Business'
        if any(k in r for k in ['operational', 'logistic', 'protocol issue', 'manufactur', 'covid', 'budget', 'site issue']): return 'Operational'
        return 'Unknown'
        
    df['termination_category'] = df['why_stopped'].apply(classify)
    return df

tidy_df = clean_and_classify_data(raw_df)
tidy_df.to_csv('tidy_trials.csv', index=False)

# Display final category breakdown
breakdown = tidy_df['termination_category'].value_counts(normalize=True) * 100
print("Termination Reason Breakdown (%):")
print(breakdown.round(1))

## Section 3: Figure Generation
**Objective:** Programmatically recreate Figure 1 and Figure 2 from Bowling et al. (2025). 

*Note: Phase 2 and Phase 3 trials are extracted using boolean masks to ensure trials flagged as both Phase 2 and Phase 3 are appropriately counted in both sub-analyses.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

# Standardize categories and colors
CATEGORIES = ["Efficacy", "Enrolment", "Operational", "Safety", "Strategy/Business", "Unknown"]
COLORS = {"Efficacy": "#3A6EA5", "Enrolment": "#88B4D6", "Operational": "#C59FC9",
          "Safety": "#C0504D", "Strategy/Business": "#E6A817", "Unknown": "#404040"}
YEARS = list(range(2015, 2025))

# Load Data
df = pd.read_csv('tidy_trials.csv', encoding="utf-8")
df["year_int"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df[df["year_int"].between(2015, 2024)].copy()
df["category"] = df["termination_category"].fillna("Unknown")
df["is_p2"] = df["phase"].astype(str).str.contains("2", na=False)
df["is_p3"] = df["phase"].astype(str).str.contains("3", na=False)

# --- FIGURE 1 ---
fig1, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(9, 7), gridspec_kw={"height_ratios": [2.5, 0.8]})
fig1.patch.set_facecolor("white")

yearly_total = df.groupby("year_int").size().reindex(YEARS, fill_value=0)
bars = ax_a.bar(YEARS, yearly_total.values, color="#3A6EA5", width=0.65, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, yearly_total.values):
    ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4, str(int(val)), ha="center", va="bottom", fontsize=9, fontweight="bold", color="#222")
ax_a.set_ylabel("Number of terminated trials", fontsize=10)
ax_a.set_xticks(YEARS)
ax_a.set_ylim(0, yearly_total.max() * 1.18)
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)
ax_a.text(-0.08, 1.02, "a", transform=ax_a.transAxes, fontsize=14, fontweight="bold", va="top")

total_all = len(df)
cat_counts = df["category"].value_counts().reindex(CATEGORIES, fill_value=0)
all_pcts = (cat_counts / total_all) * 100

left = 0
for cat in CATEGORIES:
    pct = all_pcts[cat]
    if pct == 0: continue
    ax_b.barh(0, pct, left=left, color=COLORS[cat], height=0.55, edgecolor="white", linewidth=0.8)
    if pct > 4: ax_b.text(left + pct/2, 0, f"{pct:.1f}%", ha="center", va="center", fontsize=8.5, fontweight="bold", color="white")
    left += pct

left = 0
for cat in CATEGORIES:
    pct = all_pcts[cat]
    if pct == 0: continue
    ax_b.text(left + pct/2, -0.42, cat, ha="center", va="top", fontsize=7.5, color="#333")
    left += pct

ax_b.set_xlim(0, 100)
ax_b.set_ylim(-0.8, 0.6)
ax_b.axis("off")
ax_b.text(-0.08, 1.15, "b", transform=ax_b.transAxes, fontsize=14, fontweight="bold", va="top")

fig1.suptitle("Fig. 1 | Trends in the termination of industry-sponsored phase II and\nphase III clinical trials from 2015 to 2024", fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout(h_pad=2.5)
plt.show()

# --- FIGURE 2 ---
p2, p3 = df[df["is_p2"]].copy(), df[df["is_p3"]].copy()

def calc_pcts(phase_df):
    counts = phase_df.groupby(["year_int", "category"]).size().unstack(fill_value=0).reindex(index=YEARS, columns=CATEGORIES, fill_value=0)
    totals = counts.sum(axis=1)
    pcts = counts.div(totals.replace(0, 1), axis=0).multiply(100)
    return counts, totals, pcts

p2_counts, p2_totals, p2_pcts = calc_pcts(p2)
p3_counts, p3_totals, p3_pcts = calc_pcts(p3)

fig2 = plt.figure(figsize=(14, 15))
fig2.patch.set_facecolor("white")
gs_top = fig2.add_gridspec(2, 1, top=0.97, bottom=0.52, hspace=0.38)
gs_bot = fig2.add_gridspec(2, 3, top=0.46, bottom=0.02, hspace=0.55, wspace=0.35)

ax_a, ax_b = fig2.add_subplot(gs_top[0]), fig2.add_subplot(gs_top[1])

def draw_stacked(ax, totals, pcts, label, letter):
    bottoms = np.zeros(len(YEARS))
    for cat in CATEGORIES:
        vals = pcts[cat].values
        ax.bar(YEARS, vals, bottom=bottoms, color=COLORS[cat], width=0.65, edgecolor="white", linewidth=0.4, label=cat)
        for xi, (bot, val) in enumerate(zip(bottoms, vals)):
            if val >= 0.9: ax.text(YEARS[xi], bot + val/2, f"{val:.1f}", ha="center", va="center", fontsize=5.5, color="black" if cat == "Strategy/Business" else "white", fontweight="bold")
        bottoms += vals
    for xi, tot in enumerate(totals.values): ax.text(YEARS[xi], bottoms[xi] + 0.5, str(int(tot)), ha="center", va="bottom", fontsize=8, fontweight="bold", color="#222")
    ax.set_ylabel(f"Percentage of {label}\nterminations", fontsize=10)
    ax.set_xticks(YEARS)
    ax.set_xlim(min(YEARS) - 0.6, max(YEARS) + 0.6)
    ax.set_ylim(0, 115)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.text(-0.06, 1.04, letter, transform=ax.transAxes, fontsize=13, fontweight="bold", va="top")
    if letter == "a":
        handles = [mpatches.Patch(color=COLORS[c], label=c) for c in CATEGORIES]
        ax.legend(handles=handles, loc="upper left", fontsize=8, ncol=3, framealpha=0.9, columnspacing=0.8, handlelength=1.2)

draw_stacked(ax_a, p2_totals, p2_pcts, "phase II", "a")
draw_stacked(ax_b, p3_totals, p3_pcts, "phase III", "b")

line_cats = ["Efficacy", "Enrolment", "Operational", "Safety", "Strategy/Business"]
positions = [(0,0), (0,1), (1,0), (1,1), (0,2)]
fig2.text(0.02, 0.455, "c", fontsize=13, fontweight="bold", va="top")

for cat, pos in zip(line_cats, positions):
    ax = fig2.add_subplot(gs_bot[pos])
    p2_line, p3_line = p2_pcts[cat].values, p3_pcts[cat].values
    ax.plot(YEARS, p2_line, color="#3A6EA5", linewidth=1.8, marker="o", markersize=4)
    ax.plot(YEARS, p3_line, color="#C0504D", linewidth=1.8, marker="s", markersize=4)
    ax.set_title(cat, fontsize=10, fontweight="bold")
    ax.set_ylabel("% of terminations", fontsize=8)
    ax.set_xticks(YEARS[::2])
    ymax = max(p2_line.max(), p3_line.max())
    ax.set_ylim(0, ymax * 1.35 if ymax > 0 else 1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

ax_leg = fig2.add_subplot(gs_bot[1, 2])
ax_leg.axis("off")
leg_handles = [plt.Line2D([0],[0], color="#3A6EA5", linewidth=2, marker="o", markersize=5, label="Phase II"), plt.Line2D([0],[0], color="#C0504D", linewidth=2, marker="s", markersize=5, label="Phase III")]
ax_leg.legend(handles=leg_handles, loc="center", fontsize=11, framealpha=0.9, title="Phase", title_fontsize=10)

fig2.suptitle("Fig. 2 | Trends in clinical trial terminations by phase from 2015 to 2024", fontsize=13, fontweight="bold", y=0.995)
plt.show()